In [1]:
"""
Decentralized RS — MST Item-Overlap Graph × Leave-Out-N (ML-100K)
=================================================================
Graph: Minimum Spanning Tree on cosine item-overlap dissimilarity.
Edge weight: w(u1, u2) = 1 - |I1 ∩ I2| / sqrt(|I1| * |I2|).
MST extracted via Kruskal's algorithm — one fixed topology per split.
Benchmarks: leave_out_n in [5, 10, 15, 20].
"""
from pathlib import Path
import os

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")
if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [2]:
from src.graphs import create_mst_item_overlap_graph
import warnings
warnings.filterwarnings("ignore")
import torch, time, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.optim import SGD
from tqdm import tqdm

from src.models.MatrixFactorization import UMF
from src.users import User
from src.data_utils import create_dataloader
from src.training.decentralized import (
    decentralized_train_n_epochs,
    decentralized_validate_loop,
)

torch.manual_seed(0)
np.random.seed(42)

HP = dict(
    n_factors    = 30,
    sparse       = False,
    batch_size   = 10,
    lr           = 0.03871364416669273,
    weight_decay = 0.14214480688557163,
    mom          = 0.001,
    n_epochs     = 50,
    loader_type  = "rs",
    use_dp       = True,
    dp_clip_norm = 1.0,
    dp_noise_std = 0.01,
)

VAL_FRAC = 0.20
N_LEAVE_OUT_SEQ = [5, 10, 15, 20]


def build_users(n_users, n_items, hp):
    users = {}
    for i in tqdm(range(n_users), desc="  Init users", leave=False):
        model = UMF(n_items, n_factors=hp["n_factors"], sparse=hp["sparse"])
        opt   = SGD(model.parameters(), lr=hp["lr"],
                    weight_decay=hp["weight_decay"], momentum=hp["mom"])
        users[i] = User(id=i, model=model, optimizer=opt, model_name="umf")
    return users


def dp_epsilon(sigma, n_steps, n_train, batch_size, delta=1e-5):
    q = batch_size / n_train
    return np.sqrt(2 * n_steps * np.log(1 / delta)) * q / sigma


def leave_out_n_split(ratings_df, n_leave, val_frac=VAL_FRAC, random_state=0):
    rng = np.random.default_rng(random_state)
    train_rows, val_rows, test_rows = [], [], []
    skipped_users = 0

    for uid, grp in ratings_df.groupby("user_id"):
        grp = grp.reset_index(drop=True)
        if len(grp) < n_leave:
            skipped_users += 1
            continue
        test_idx   = grp.index[-n_leave:]
        train_pool = grp.drop(index=test_idx)
        if len(train_pool) == 0:
            skipped_users += 1
            continue
        n_val = max(1, int(np.floor(len(train_pool) * val_frac)))
        val_idx_local = rng.choice(len(train_pool), size=n_val, replace=False)
        val_mask = np.zeros(len(train_pool), dtype=bool)
        val_mask[val_idx_local] = True
        val_rows.append(train_pool.iloc[val_mask])
        train_rows.append(train_pool.iloc[~val_mask])
        test_rows.append(grp.iloc[test_idx])

    train_df = pd.concat(train_rows, ignore_index=True)
    val_df   = pd.concat(val_rows,   ignore_index=True)
    test_df  = pd.concat(test_rows,  ignore_index=True)

    train_items = set(train_df.item_id.unique())
    val_df  = val_df[val_df.item_id.isin(train_items)].reset_index(drop=True)
    test_df = test_df[test_df.item_id.isin(train_items)].reset_index(drop=True)

    all_users = np.unique(pd.concat([train_df, val_df, test_df]).user_id.values)
    all_items = np.unique(pd.concat([train_df, val_df, test_df]).item_id.values)
    u2i = {u: i for i, u in enumerate(all_users)}
    a2i = {a: i for i, a in enumerate(all_items)}
    for df in [train_df, val_df, test_df]:
        df["user_id"] = df["user_id"].map(u2i)
        df["item_id"] = df["item_id"].map(a2i)

    meta = dict(n_leave=n_leave, n_users=len(all_users), n_items=len(all_items),
                skipped_users=skipped_users, n_train=len(train_df),
                n_val=len(val_df), n_test=len(test_df))
    return train_df, val_df, test_df, meta


def run_experiment_mst(label, train_df, val_df, test_df, n_items, hp):
    print(f"\n{'─'*60}")
    print(f"  {label}  |  "
          f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
    print(f"{'─'*60}")

    n_users = train_df["user_id"].nunique()

    train_loader = create_dataloader(df=train_df, dl_type=hp["loader_type"],
                                     batch_size=hp["batch_size"])
    val_loader   = create_dataloader(df=val_df,   dl_type=hp["loader_type"])
    test_loader  = create_dataloader(df=test_df,  dl_type=hp["loader_type"])

    users = build_users(n_users, n_items, hp)

    user_items_dict = {
        uid: set(grp["item_id"].tolist())
        for uid, grp in train_df.groupby("user_id")
    }
    graph = create_mst_item_overlap_graph(
        n_users=n_users, user_items_dict=user_items_dict
    )
    print(f"  Graph: {graph.number_of_nodes()} nodes, "
          f"{graph.number_of_edges()} edges  "
          f"(avg deg: {2*graph.number_of_edges()/max(n_users,1):.1f})")

    torch.manual_seed(0)
    t0 = time.time()
    train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users, train_loader=train_loader, val_loader=val_loader,
        epochs=hp["n_epochs"], graph=graph,
    )
    elapsed = time.time() - t0

    test_rmse  = float(decentralized_validate_loop(users, test_loader))
    best_val   = float(min(val_losses))
    best_epoch = int(np.argmin(val_losses)) + 1
    epochs_run = len(train_losses)

    param_bytes       = hp["n_factors"] * 4
    total_commute     = int(sum(commutes["commute"]))
    comm_cost_mb      = round(total_commute * param_bytes / 1e6, 3)
    avg_commute_epoch = round(total_commute / max(epochs_run, 1), 1)
    eps = dp_epsilon(hp["dp_noise_std"], epochs_run * len(train_loader),
                     len(train_df), hp["batch_size"])

    result = dict(
        label=label,
        n_train=len(train_df), n_val=len(val_df), n_test=len(test_df),
        n_users=n_users, n_items=n_items,
        n_edges=graph.number_of_edges(),
        avg_degree=round(2*graph.number_of_edges()/max(n_users,1), 2),
        test_rmse=round(test_rmse, 6),
        best_val_loss=round(best_val, 6),
        best_epoch=best_epoch, epochs_run=epochs_run,
        train_losses=[round(x, 6) for x in train_losses],
        val_losses=[round(x, 6) for x in val_losses],
        time_per_epoch=[round(x, 3) for x in time_per_epoch],
        commutes=commutes, total_commute=total_commute,
        comm_cost_mb=comm_cost_mb, avg_commute_epoch=avg_commute_epoch,
        elapsed_s=round(elapsed, 2),
        dp_epsilon=round(eps, 4), dp_noise_std=hp["dp_noise_std"],
    )
    print(f"  ✓  Test RMSE: {test_rmse:.4f}  |  epoch {best_epoch}"
          f"  |  {comm_cost_mb} MB  |  ε={eps:.2f}  |  {elapsed:.1f}s")
    return result


In [3]:
column_names = ["user_id", "item_id", "rating", "timestamp"]
ratings = pd.read_csv("dataset/u.data", sep="\t",
                      names=column_names).iloc[:, 0:3]
print(f"Raw ratings: {len(ratings):,}  |  "
      f"Users: {ratings['user_id'].nunique()}  |  "
      f"Items: {ratings['item_id'].nunique()}")

all_results = []

for n_leave in N_LEAVE_OUT_SEQ:
    train_df, val_df, test_df, meta = leave_out_n_split(
        ratings, n_leave=n_leave, val_frac=VAL_FRAC
    )
    print(f"\nLeave-{n_leave}: users={meta['n_users']} items={meta['n_items']} "
          f"train={meta['n_train']:,} val={meta['n_val']:,} test={meta['n_test']:,}")

    label = f"mst_leave{n_leave}"
    res = run_experiment_mst(
        label=label, train_df=train_df, val_df=val_df, test_df=test_df,
        n_items=meta["n_items"], hp=HP,
    )
    res["n_leave"]       = n_leave
    res["skipped_users"] = meta["skipped_users"]
    all_results.append(res)


Raw ratings: 100,000  |  Users: 943  |  Items: 1682

Leave-5: users=943 items=1650 train=76,595 val=18,653 test=4,712

────────────────────────────────────────────────────────────
  mst_leave5  |  train=76595  val=18653  test=4712
────────────────────────────────────────────────────────────


  Graph: 943 nodes, 942 edges  (avg deg: 2.0)


  0%|          | 0/76595 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.5852 | Validation Loss: 1.2230 | Time Elapsed: 39.025383 sec |Commute: 185954 | Commute 
Cost: 3917834250

  0%|          | 0/76595 [00:00<?, ?it/s]

...

Early stopping.

Total time elapsed: 461.4834864999866

  ✓  Test RMSE: 1.2563  |  epoch 12  |  268.169 MB  |  ε=65.84  |  461.5s

Leave-20: users=911 items=1630 train=65,190 val=15,903 test=18,192

────────────────────────────────────────────────────────────
  mst_leave20  |  train=65190  val=15903  test=18192
────────────────────────────────────────────────────────────


  Graph: 887 nodes, 886 edges  (avg deg: 2.0)


KeyError: 899

In [6]:
print("\n" + "═"*80)
print(f"  {'Label':<14} {'TrainN':>7} {'TestN':>7} {'TestRMSE':>10} {'CommMB':>9} {'ε':>7}")
print("═"*80)
for r in all_results:
    print(f"  {r['label']:<14} {r['n_train']:>7} {r['n_test']:>7}"
          f" {r['test_rmse']:>10.4f} {r['comm_cost_mb']:>9.2f} {r['dp_epsilon']:>7.2f}")
print("═"*80)
out = Path("mst_overlap_leaveout_results.json")
out.write_text(json.dumps(all_results, indent=2))
print(f"\nResults saved → {out}")



════════════════════════════════════════════════════════════════════════════════
  Label           TrainN   TestN   TestRMSE    CommMB       ε
════════════════════════════════════════════════════════════════════════════════
  mst_leave5       76595    4712     0.9741    334.72   67.15
  mst_leave10      72823    9423     1.0846    296.23   66.53
  mst_leave15      69051   14129     1.2563    268.17   65.84
════════════════════════════════════════════════════════════════════════════════


TypeError: Object of type float32 is not JSON serializable